In [1]:
import pickle
import pandas as pd
import numpy as np
import torch
import transformers
import bitsandbytes as bnb
import random

random_state = 1

random.seed(random_state)

In [2]:
def euclid(list1, list2):
    return sum((p-q)**2 for p, q in zip(list1, list2)) ** .5

In [3]:
with open('df_annotations_embeddings.pkl', 'rb') as f:
    df = pickle.load(f)
df

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
0,A2008_Commission of the European Communities v...,1,prem,A1,must held first Article 4c CS prohibits granti...,L,Rule,"[-0.01507568359375, -1.73828125, 1.4287109375,..."
1,A2008_Commission of the European Communities v...,1,prem,A2,Also clear consistent caselaw Articles 4 CS 67...,L,"[Itpr, Prec, Rule]","[0.93017578125, -1.0146484375, 0.0575256347656..."
2,A2008_Commission of the European Communities v...,1,prem,A3,Court deduces Article 67 CS covers general mea...,L,"[Prec, Rule]","[0.195556640625, -1.734375, -0.1546630859375, ..."
3,A2008_Commission of the European Communities v...,1,prem,A4,Court also held action taken Article 67 CS can...,L,"[Prec, Rule]","[0.404296875, -0.2352294921875, 0.34326171875,..."
4,A2008_Commission of the European Communities v...,1,prem,A5,Court held particular inconceivable authors EC...,L,"[Itpr, Prec, Rule]","[1.0849609375, -0.98828125, 0.39453125, 0.8774..."
...,...,...,...,...,...,...,...,...
2530,R2021_World Duty Free v,5,prem,H7,question matter dialogue Spanish authorities C...,F,NaN,"[2.109375, -1.3642578125, -0.62744140625, 2.50..."
2531,R2021_World Duty Free v,5,prem,H8,present case action first instance WDFG compla...,F,NaN,"[0.880859375, -0.65966796875, -0.67626953125, ..."
2532,R2021_World Duty Free v,5,prem,H9,regard General Court recalled paragraph 212 ju...,F,NaN,"[2.0234375, -1.150390625, -0.70947265625, 1.59..."
2533,R2021_World Duty Free v,5,prem,H10,General Court err law concluding therefrom par...,F,NaN,"[1.6494140625, -1.5732421875, -0.857421875, 1...."


In [4]:
df_train = df[df['Split'] != 2]
df_train = df_train.dropna(subset = ['Scheme'])
df_train

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
0,A2008_Commission of the European Communities v...,1,prem,A1,must held first Article 4c CS prohibits granti...,L,Rule,"[-0.01507568359375, -1.73828125, 1.4287109375,..."
1,A2008_Commission of the European Communities v...,1,prem,A2,Also clear consistent caselaw Articles 4 CS 67...,L,"[Itpr, Prec, Rule]","[0.93017578125, -1.0146484375, 0.0575256347656..."
2,A2008_Commission of the European Communities v...,1,prem,A3,Court deduces Article 67 CS covers general mea...,L,"[Prec, Rule]","[0.195556640625, -1.734375, -0.1546630859375, ..."
3,A2008_Commission of the European Communities v...,1,prem,A4,Court also held action taken Article 67 CS can...,L,"[Prec, Rule]","[0.404296875, -0.2352294921875, 0.34326171875,..."
4,A2008_Commission of the European Communities v...,1,prem,A5,Court held particular inconceivable authors EC...,L,"[Itpr, Prec, Rule]","[1.0849609375, -0.98828125, 0.39453125, 0.8774..."
...,...,...,...,...,...,...,...,...
2520,R2021_World Duty Free v,5,prem,G4,accordance settled caselaw appeal plea directe...,L,Prec,"[0.5693359375, -1.30078125, 0.0073127746582031..."
2521,R2021_World Duty Free v,5,prem,G5,present case even fifth part single ground app...,L,Prec,"[0.484130859375, -0.362548828125, -0.387695312..."
2526,R2021_World Duty Free v,5,prem,H4,pointed paragraph 146 appeal plea directed gro...,"[F, L]",Itpr,"[0.34423828125, -0.83349609375, -0.04077148437..."
2528,R2021_World Duty Free v,5,prem,H6,regard true decision adopts end examination Co...,L,Princ,"[1.3798828125, -0.8056640625, 0.01039886474609..."


In [5]:
def contains_str(value, string):
    return string in value

In [6]:
df_train_Rule = df_train[df_train['Scheme'].apply(lambda x: contains_str(x, 'Rule'))]
df_train_Rule

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
0,A2008_Commission of the European Communities v...,1,prem,A1,must held first Article 4c CS prohibits granti...,L,Rule,"[-0.01507568359375, -1.73828125, 1.4287109375,..."
1,A2008_Commission of the European Communities v...,1,prem,A2,Also clear consistent caselaw Articles 4 CS 67...,L,"[Itpr, Prec, Rule]","[0.93017578125, -1.0146484375, 0.0575256347656..."
2,A2008_Commission of the European Communities v...,1,prem,A3,Court deduces Article 67 CS covers general mea...,L,"[Prec, Rule]","[0.195556640625, -1.734375, -0.1546630859375, ..."
3,A2008_Commission of the European Communities v...,1,prem,A4,Court also held action taken Article 67 CS can...,L,"[Prec, Rule]","[0.404296875, -0.2352294921875, 0.34326171875,..."
4,A2008_Commission of the European Communities v...,1,prem,A5,Court held particular inconceivable authors EC...,L,"[Itpr, Prec, Rule]","[1.0849609375, -0.98828125, 0.39453125, 0.8774..."
...,...,...,...,...,...,...,...,...
2406,R2021_World Duty Free v,5,prem,B11,regard Court held numerous occasions objective...,L,"[Prec, Rule]","[0.91455078125, -0.69482421875, -0.76904296875..."
2415,R2021_World Duty Free v,5,prem,B17,regard Court Justice points reviewing legality...,L,"[Prec, Rule]","[1.5009765625, -1.8251953125, -0.474853515625,..."
2416,R2021_World Duty Free v,5,prem,B18,Article 264 TFEU provides action well founded ...,L,Rule,"[0.237060546875, -1.1904296875, 1.1767578125, ..."
2452,R2021_World Duty Free v,5,prem,B53,Admittedly appellants rightly argued apparent ...,"[F, L]","[Prec, Rule]","[0.431884765625, -0.5126953125, 0.21923828125,..."


In [7]:
df_train_Itpr = df_train[df_train['Scheme'].apply(lambda x: contains_str(x, 'Itpr'))]
df_train_Itpr

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
1,A2008_Commission of the European Communities v...,1,prem,A2,Also clear consistent caselaw Articles 4 CS 67...,L,"[Itpr, Prec, Rule]","[0.93017578125, -1.0146484375, 0.0575256347656..."
4,A2008_Commission of the European Communities v...,1,prem,A5,Court held particular inconceivable authors EC...,L,"[Itpr, Prec, Rule]","[1.0849609375, -0.98828125, 0.39453125, 0.8774..."
7,A2008_Commission of the European Communities v...,1,prem,A8,Finally State aid granted undertaking falling ...,L,Itpr,"[-0.6279296875, -1.154296875, 1.5390625, 0.588..."
8,A2008_Commission of the European Communities v...,1,prem,A9,light foregoing must consequently held Article...,L,"[Itpr, Rule]","[0.65283203125, -1.828125, 0.75, 1.142578125, ..."
9,A2008_Commission of the European Communities v...,1,prem,A10,Contrary German Government claims interpretati...,"[F, L]","[Itpr, Rule]","[0.460693359375, -0.8759765625, -0.6455078125,..."
...,...,...,...,...,...,...,...,...
2499,R2021_World Duty Free v,5,prem,D14,regard must borne mind accordance caselaw ment...,L,"[Itpr, Prec]","[1.9736328125, -1.0966796875, 0.06695556640625..."
2507,R2021_World Duty Free v,5,prem,E2,party entitled put forward pleas arguments ari...,L,Itpr,"[0.4736328125, -0.90234375, 0.394287109375, 2...."
2508,R2021_World Duty Free v,5,prem,E3,appellants therefore entitled call question fi...,L,Itpr,"[1.095703125, -1.23828125, -0.1693115234375, 1..."
2526,R2021_World Duty Free v,5,prem,H4,pointed paragraph 146 appeal plea directed gro...,"[F, L]",Itpr,"[0.34423828125, -0.83349609375, -0.04077148437..."


In [8]:
df_train_Aut = df_train[df_train['Scheme'].apply(lambda x: contains_str(x, 'Aut'))]
df_train_Aut

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
316,A2013_European Commission v Ireland and Others,3,prem,A9,Yet submitted Commission stated Advocate Gener...,F,Aut,"[1.9296875, -0.994140625, -1.013671875, 1.4736..."
402,A2016_European Commission v Aer Lingus Ltd and...,3,prem,A6,Advocate General observed essence point 41 Opi...,F,Aut,"[1.861328125, -0.7568359375, 0.1915283203125, ..."
435,A2016_European Commission v Aer Lingus Ltd and...,3,prem,D11,follows Advocate General observed essence poin...,L,"[Aut, Itpr]","[0.7578125, -1.0390625, -0.47314453125, 0.9809..."
520,A2016_European_Commission_v_World_Duty_Free,4,prem,B1,regards caselaw relating aid exports relied co...,"[F, L]","[Aut, Itpr, Prec]","[0.56640625, -0.97802734375, -0.8818359375, 0...."
524,A2016_European_Commission_v_World_Duty_Free,4,prem,B5,observed essence Advocate General points 133 1...,L,Aut,"[1.6123046875, -0.6708984375, -0.1895751953125..."
569,A2017_European Commission v Italian Republic_DT,4,prem,A10,addition Advocate General stated point 38 opin...,F,Aut,"[1.5224609375, -0.841796875, -0.388427734375, ..."
609,A2017_European Commission v Italian Republic_DT,4,prem,B37,apparent paragraphs 46 52 observed essence Adv...,L,"[Aut, Itpr]","[1.330078125, -0.74267578125, 0.00800323486328..."
755,A2018_Dirk Andres v European Commission,5,prem,A13,also noted essence Advocate General points 57 ...,"[F, L]","[Aut, Prec, Rule]","[1.1591796875, -0.7392578125, -0.53076171875, ..."
798,A2018_Dirk Andres v European Commission,5,prem,C22,Consequently also noted essence Advocate Gener...,L,Aut,"[1.205078125, -0.26904296875, -0.316650390625,..."
809,A2018_Dirk Andres v European Commission,5,prem,C22|C23|C32,also noted essence Advocate General point 109 ...,L,"[Aut, Itpr]","[1.5673828125, 0.0211334228515625, -0.61865234..."


In [9]:
df_train_Class = df_train[df_train['Scheme'].apply(lambda x: contains_str(x, 'Class'))]
df_train_Class

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
120,A2009_Commission of the European Communities v...,1,prem,A2,relation first condition set therein settled c...,L,"[Class, Prec]","[0.9697265625, -1.6416015625, 0.5419921875, 1...."
121,A2009_Commission of the European Communities v...,1,prem,A3,present case Court First Instance held paragra...,F,"[Class, Prec, Rule]","[1.4775390625, -0.89404296875, -0.7373046875, ..."
463,A2016_European_Commission_v_World_Duty_Free,4,prem,A1,First must recalled according Court’s settled ...,L,"[Class, Prec, Rule]","[0.418701171875, -0.712890625, -0.207397460937..."
464,A2016_European_Commission_v_World_Duty_Free,4,prem,A1bis,First must intervention State State resources,L,Class,"[1.8837890625, -1.041015625, -0.82080078125, 2..."
465,A2016_European_Commission_v_World_Duty_Free,4,prem,A1ter,Second intervention must liable affect trade M...,L,Class,"[1.5693359375, -1.37890625, -0.32177734375, 2...."
467,A2016_European_Commission_v_World_Duty_Free,4,prem,A1quater,Fourth must distort threaten distort competiti...,L,"[Class, Prec]","[0.681640625, -0.297119140625, -0.7919921875, ..."
612,A2017_European Commission v TV2_Danmark A_S,5,prem,A1,accordance Court’s settled caselaw classificat...,L,"[Class, Prec, Rule]","[0.7919921875, -1.1328125, -0.8232421875, 0.68..."
613,A2017_European Commission v TV2_Danmark A_S,5,prem,A2,provision sets four conditions,L,Class,"[0.60986328125, -0.7099609375, 0.8125, 1.06640..."
614,A2017_European Commission v TV2_Danmark A_S,5,prem,A2bis,First must intervention State State resources,L,Class,"[1.8837890625, -1.041015625, -0.82080078125, 2..."
615,A2017_European Commission v TV2_Danmark A_S,5,prem,A2ter,Second intervention must liable affect trade M...,L,Class,"[1.5693359375, -1.37890625, -0.32177734375, 2...."


In [10]:
df_train_Prec = df_train[df_train['Scheme'].apply(lambda x: contains_str(x, 'Prec'))]
df_train_Prec

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
1,A2008_Commission of the European Communities v...,1,prem,A2,Also clear consistent caselaw Articles 4 CS 67...,L,"[Itpr, Prec, Rule]","[0.93017578125, -1.0146484375, 0.0575256347656..."
2,A2008_Commission of the European Communities v...,1,prem,A3,Court deduces Article 67 CS covers general mea...,L,"[Prec, Rule]","[0.195556640625, -1.734375, -0.1546630859375, ..."
3,A2008_Commission of the European Communities v...,1,prem,A4,Court also held action taken Article 67 CS can...,L,"[Prec, Rule]","[0.404296875, -0.2352294921875, 0.34326171875,..."
4,A2008_Commission of the European Communities v...,1,prem,A5,Court held particular inconceivable authors EC...,L,"[Itpr, Prec, Rule]","[1.0849609375, -0.98828125, 0.39453125, 0.8774..."
5,A2008_Commission of the European Communities v...,1,prem,A6,Court also held first indent Article 672 CS de...,L,"[Prec, Rule]","[0.05633544921875, -1.2216796875, 0.4230957031..."
...,...,...,...,...,...,...,...,...
2502,R2021_World Duty Free v,5,prem,D16,Court Justice held measure measure issue desig...,L,Prec,"[1.08203125, -1.0966796875, -0.65283203125, 0...."
2514,R2021_World Duty Free v,5,prem,F2,arguments cannot however upheld since apparent...,"[F, L]","[Prec, Princ]","[0.37890625, 0.330322265625, 0.5732421875, 0.6..."
2515,R2021_World Duty Free v,5,prem,F3,stage Member State thus called demonstrate dif...,L,"[Prec, Princ]","[1.11328125, -1.2724609375, 0.088134765625, 0...."
2520,R2021_World Duty Free v,5,prem,G4,accordance settled caselaw appeal plea directe...,L,Prec,"[0.5693359375, -1.30078125, 0.0073127746582031..."


In [11]:
df_test = df[df['Split'] == 2]
df_test = df_test.dropna(subset = ['Scheme'])
df_test

,Document,Split,Name,Id,Text,Type,Scheme,embeddings
172,A2010_NDSHT Nya Destination Stockholm Hotell &...,2,prem,A2,regard must observed follows Article 58 Statut...,L,"[Prec, Rule]","[0.5068359375, -1.6494140625, -0.43408203125, ..."
177,A2010_NDSHT Nya Destination Stockholm Hotell &...,2,prem,B2,regard Court repeatedly held action annulment ...,L,"[Prec, Rule]","[1.076171875, -0.91064453125, -0.53271484375, ..."
178,A2010_NDSHT Nya Destination Stockholm Hotell &...,2,prem,B3,follows also settled caselaw concerning admiss...,L,Prec,"[0.90185546875, -1.087890625, -0.5048828125, 1..."
179,A2010_NDSHT Nya Destination Stockholm Hotell &...,2,prem,B4,contrast form act decision adopted principle i...,L,Itpr,"[0.92431640625, -0.471435546875, -0.3063964843..."
180,A2010_NDSHT Nya Destination Stockholm Hotell &...,2,prem,B5,therefore principle irrelevant classification ...,L,Itpr,"[0.7216796875, -0.82373046875, 0.39501953125, ..."
...,...,...,...,...,...,...,...,...
2123,R2017_European Commission v Frucona Košice a,2,prem,D13,connection also borne mind lawfulness decision...,L,Prec,"[0.83251953125, -0.8701171875, -0.16796875, 1...."
2124,R2017_European Commission v Frucona Košice a,2,prem,D14,However information ‘available’ Commission inc...,L,"[Itpr, Prec]","[1.4931640625, -1.1728515625, -0.70751953125, ..."
2126,R2017_European Commission v Frucona Košice a,2,prem,D16,Paragraphs 180 213 235 judgment appeal Commiss...,F,Itpr,"[0.55419921875, -1.5498046875, -0.5234375, 1.4..."
2130,R2017_European Commission v Frucona Košice a,2,prem,D20,Advocate General noted paragraphs 125 131 Opin...,F,Aut,"[1.2119140625, -1.08984375, 0.0924072265625, -..."


In [12]:
device = torch.device('cuda')
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [13]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16,
                  "quantization_config": {"load_in_4bit": True}},
    device_map="cuda:0"
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


In [14]:
def test(template, nexamples, df_train = df_train, df_train_Aut = df_train_Aut, df_train_Class = df_train_Class, df_train_Itpr = df_train_Itpr, df_train_Prec = df_train_Prec, df_train_Rule = df_train_Rule):

    count = 0
    autTP = 0
    autFN = 0
    autFP = 0
    classTP = 0
    classFN = 0
    classFP = 0
    itprTP = 0
    itprFN = 0
    itprFP = 0
    precTP = 0
    precFN = 0
    precFP = 0
    #princTP = 0
    #princFN = 0
    #princFP = 0
    ruleTP = 0
    ruleFN = 0
    ruleFP = 0

    for i in range(len(df_test)):

        query = 'Text: '+ df_test.iloc[i]['Text']+'\n'+'Answer: '

        df_train = df_train.sample(frac = 1, random_state = random_state)
        df_train_Aut = df_train_Aut.sample(frac = 1, random_state = random_state)
        df_train_Class = df_train_Class.sample(frac = 1, random_state = random_state)
        df_train_Itpr = df_train_Itpr.sample(frac = 1, random_state = random_state)
        df_train_Prec = df_train_Prec.sample(frac = 1, random_state = random_state)
        df_train_Rule = df_train_Rule.sample(frac = 1, random_state = random_state)

        examples = list(range((nexamples+4)//5))

        examples_text = []


        for el in list(examples):
            examples_text.append('Text: '+ df_train_Aut.iloc[el]['Text'] + '\n' + 'Answer: '+ str(df_train_Aut.iloc[el]['Scheme'])+'\n\n')
            examples_text.append('Text: '+ df_train_Class.iloc[el]['Text'] + '\n' + 'Answer: '+ str(df_train_Class.iloc[el]['Scheme'])+'\n\n')
            examples_text.append('Text: '+ df_train_Itpr.iloc[el]['Text'] + '\n' + 'Answer: '+ str(df_train_Itpr.iloc[el]['Scheme'])+'\n\n')
            examples_text.append('Text: '+ df_train_Prec.iloc[el]['Text'] + '\n' + 'Answer: '+ str(df_train_Prec.iloc[el]['Scheme'])+'\n\n')
            examples_text.append('Text: '+ df_train_Rule.iloc[el]['Text'] + '\n' + 'Answer: '+ str(df_train_Rule.iloc[el]['Scheme'])+'\n\n')
            
        if (nexamples) % 2 == 1:
                examples_text.append('Text: '+ df_train.iloc[0]['Text'] + '\n' + 'Answer: '+ str(df_train.iloc[0]['Scheme'])+'\n\n')
            
        random.shuffle(examples_text)

        examples_text = examples_text[:nexamples]
        
        text_examples = ''
        for el in examples_text:
            text_examples += el
            
        print('query:\n'+ query,'\n')
        print('retrieved examples:\n\n'+ text_examples,'\n\n')

        messages = [{'role': 'system', 'content': template},{"role": "user", "content": text_examples},{"role": "user", "content": query}]

        outputs = pipeline(
            messages,
            max_new_tokens=10,
            do_sample = False
        )
        final_output = outputs[0]['generated_text'][-1]['content']
        print('LLM output:', final_output)
        print('ground truth:', df_test.iloc[i]['Scheme'])
        print(200*'-')

        if 'Aut' in final_output and 'Aut' in df_test.iloc[i]['Scheme']:
            autTP += 1
        if 'Aut' in final_output and 'Aut' not in df_test.iloc[i]['Scheme']:
            autFP += 1
        if 'Aut' not in final_output and 'Aut' in df_test.iloc[i]['Scheme']:
            autFN += 1
        
        if 'Class' in final_output and 'Class' in df_test.iloc[i]['Scheme']:
            classTP += 1
        if 'Class' in final_output and 'Class' not in df_test.iloc[i]['Scheme']:
            classFP += 1
        if 'Class' not in final_output and 'Class' in df_test.iloc[i]['Scheme']:
            classFN += 1

        if 'Itpr' in final_output and 'Itpr' in df_test.iloc[i]['Scheme']:
            itprTP += 1
        if 'Itpr' in final_output and 'Itpr' not in df_test.iloc[i]['Scheme']:
            itprFP += 1
        if 'Itpr' not in final_output and 'Itpr' in df_test.iloc[i]['Scheme']:
            itprFN += 1

        if 'Prec' in final_output and 'Prec' in df_test.iloc[i]['Scheme']:
            precTP += 1
        if 'Prec' in final_output and 'Prec' not in df_test.iloc[i]['Scheme']:
            precFP += 1
        if 'Prec' not in final_output and 'Prec' in df_test.iloc[i]['Scheme']:
            precFN += 1

        if 'Rule' in final_output and 'Rule' in df_test.iloc[i]['Scheme']:
            ruleTP += 1
        if 'Rule' in final_output and 'Rule' not in df_test.iloc[i]['Scheme']:
            ruleFP += 1
        if 'Rule' not in final_output and 'Rule' in df_test.iloc[i]['Scheme']:
            ruleFN += 1

    if autTP == 0:
        autF1 = 0
    else:
        autPREC = autTP/(autTP+autFP)
        autREC = autTP/(autTP+autFN)
        autF1 = 2/((1/autPREC)+(1/autREC))

    if classTP == 0:
        classF1 = 0
    else:
        classPREC = classTP/(classTP+classFP)
        classREC = classTP/(classTP+classFN)
        classF1 = 2/((1/classPREC)+(1/classREC))

    if itprTP == 0:
        itprF1 = 0
    else:
        itprPREC = itprTP/(itprTP+itprFP)
        itprREC = itprTP/(itprTP+itprFN)
        itprF1 = 2/((1/itprPREC)+(1/itprREC))

    if precTP == 0:
        precF1 = 0
    else:
        precPREC = precTP/(precTP+precFP)
        precREC = precTP/(precTP+precFN)
        precF1 = 2/((1/precPREC)+(1/precREC))

    if ruleTP == 0:
        ruleF1 = 0
    else:
        rulePREC = ruleTP/(ruleTP+ruleFP)
        ruleREC = ruleTP/(ruleTP+ruleFN)
        ruleF1 = 2/((1/rulePREC)+(1/ruleREC))

    print('Authoritative F1:', autF1)
    print('Verbal Classification F1:', classF1)
    print('Interpretation F1:', itprF1)
    print('Precedent F1:', precF1)
    print('Rule F1:', ruleF1)
    print()


    print('macro F1:', (autF1 + classF1 + itprF1 + precF1 + ruleF1)/5)
    print('macro F1 (reliable):', (autF1 + precF1 + ruleF1)/3)
            


In [15]:
template = '''Classify the following legal premise as one or more of the following argumentative schemes: Rule, Prec, Class, Itpr, Aut. Rule: whether there is an explicit or implicit reference to an article of law or citation of the text of a certain article. Prec: whether there is a reference to a previous ruling of the Supreme Court or the Court of Justice of the European Union. Class: if there is a definition of a legal concept or its constituent elements. Itpr: if there is reference to one of the interpretative criteria contained in Article 12 of the prelegislations (literal, teleological, psychological, systematic) to the Civil Code. Aut: if there is a reference to an indication by an authority (e.g. an opinion of the Advocate General). The expected output is a list with all applicable labels. For example: ['Prec', 'Aut', 'Rule']. Only reply with the list of labels.'''

In [16]:
test(template, 5)

C:\Users\alfio\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\generation\configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\alfio\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\generation\configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


query:
Text: regard must observed follows Article 58 Statute Court Justice conjunction Article 1132 Rules Procedure Court Justice appeal appellant may put forward relevant argument provided subjectmatter proceedings General Court changed appeal Case C‑22905 P PKK KNK v Council 2007 ECR I‑439 paragraph 66 Case C‑806 P Herrero Romeu v Commission 2007 ECR I‑10333 paragraph 32
Answer:  

retrieved examples:

Text: contrary exclude priori possibility case present trade union could show party concerned within meaning Article 882 EC relying role collective negotiations effects role national tax measures regarded Commission aid compatible common market would liable undermine social policy objectives induced Court exclude collective agreement issue Albany application Article 851 Treaty
Answer: ['Itpr', 'Rule']

Text: Next since outside spheres EU tax law harmonised Member State concerned defines exercising exclusive competence matter direct taxation characteristics constituting tax determinatio

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Aut']
ground truth: ['Prec', 'Rule']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: regard Court repeatedly held action annulment purposes Article 230 EC must available acts adopted institutions whatever nature form intended legal effects capable affecting interests applicant bringing distinct change legal position see inter alia Athinaïki Techniki v Commission paragraph 29 caselaw cited Case C‑36208 P Internationaler Hilfsfonds v Commission 2010 ECR I‑0000 paragraph 51
Answer:  

retrieved examples:

Text: Advocate General observes points 107 110 Opinion principle ‘no one obliged impossible’ among general principles EU law see effect order 3 March 2016 Daimler C‑17915 EUC2016134 paragraph 42
Answer: ['Aut', 'Prec', 'Princ']

Text: Third must confer selective advantage recipient
Answer: Class

Text: 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

LLM output: ['Rule']
ground truth: ['Itpr', 'Prec']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Paragraphs 180 213 235 judgment appeal Commission’s objections referred paragraph 64 present judgment directed read legal context referred General Court imply new requirements incompatible caselaw Court Justice
Answer:  

retrieved examples:

Text: conclusion supported fact since Community economic also social purpose rights provisions Treaty State aid competition must balanced appropriate objectives pursued social policy include clear first paragraph Article 136 EC inter alia improved living working conditions make possible harmonisation improvement maintained proper social protection dialogue management labour see effect respect Treaty provisions freedom establishment Case C‑43805 International Transport Workers’ Federatio

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Rule']
ground truth: Itpr
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Advocate General noted paragraphs 125 131 Opinion General Court merely noted paragraphs 191 195 198 199 judgment appeal internal contradictions decision issue made findings fact according none evidence administrative file able substantiate liquidation factors used Commission
Answer:  

retrieved examples:

Text: Furthermore aid granted Member State strengthens position undertaking compared undertakings competing trade within European Union undertakings must regarded affected aid judgment 10 January 2006 Cassa di Risparmio di Firenze Others C‑22204 EUC20068 paragraph 141
Answer: Prec

Text: Therefore arguments put forward Orange appeal ineffective even well founded could result judgment appeal set aside
Answer: Aut

Text: contrar

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Aut', 'Rule']
ground truth: Aut
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: considerations far concern information normally prudent diligent private creditor situation close possible local tax office could priori ignore capable justifying General Court’s decision Commission failed take consideration relevant information see effect judgment 24 January 2013 Frucona Košice v Commission C‑7311 P EUC201332 paragraphs 77 78 81
Answer:  

retrieved examples:

Text: Thus according settled caselaw jurisdiction Court Justice appeal limited review findings law pleas arguments debated General Court
Answer: Prec

Text: First must recalled according Court’s settled caselaw classification national measure ‘State aid’ within meaning Article 1071 TFEU requires following conditions fulfilled
Answer: ['Class', 'Prec', 'Rule

In [17]:
test(template, 10)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


query:
Text: regard must observed follows Article 58 Statute Court Justice conjunction Article 1132 Rules Procedure Court Justice appeal appellant may put forward relevant argument provided subjectmatter proceedings General Court changed appeal Case C‑22905 P PKK KNK v Council 2007 ECR I‑439 paragraph 66 Case C‑806 P Herrero Romeu v Commission 2007 ECR I‑10333 paragraph 32
Answer:  

retrieved examples:

Text: First must recalled according Court’s settled caselaw classification national measure ‘State aid’ within meaning Article 1071 TFEU requires following conditions fulfilled
Answer: ['Class', 'Prec', 'Rule']

Text: Judicial review compliance European Union legal order ensured seen Article 191 TEU Court Justice courts tribunals Member States
Answer: Rule

Text: contrary exclude priori possibility case present trade union could show party concerned within meaning Article 882 EC relying role collective negotiations effects role national tax measures regarded Commission aid compatible c

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Rule']
ground truth: ['Prec', 'Rule']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: regard Court repeatedly held action annulment purposes Article 230 EC must available acts adopted institutions whatever nature form intended legal effects capable affecting interests applicant bringing distinct change legal position see inter alia Athinaïki Techniki v Commission paragraph 29 caselaw cited Case C‑36208 P Internationaler Hilfsfonds v Commission 2010 ECR I‑0000 paragraph 51
Answer:  

retrieved examples:

Text: must observed outset Advocate General stated point 47 Opinion requirement selectivity Article 1071 TFEU must clearly distinguished concomitant detection economic advantage Commission identified advantage understood broad sense arising directly indirectly particular measure also required establish

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

LLM output: ['Rule']
ground truth: ['Itpr', 'Prec']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Paragraphs 180 213 235 judgment appeal Commission’s objections referred paragraph 64 present judgment directed read legal context referred General Court imply new requirements incompatible caselaw Court Justice
Answer:  

retrieved examples:

Text: Contrary arguments Commission conclusion called question judgments 28 April 2015 TL Sugars Sidul Açúcares v Commission C‑45613 P EUC2015284 17 September 2015 Confederazione Cooperative Italiane Others v Anicav Others C‑45513 P C‑45713 P C‑46013 P published EUC2015616
Answer: Prec

Text: Second intervention must liable affect trade Member States
Answer: Class

Text: end national court may decide suspend implementation measure question order recovery sums already paid
Answer: Itpr



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Aut']
ground truth: Itpr
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Advocate General noted paragraphs 125 131 Opinion General Court merely noted paragraphs 191 195 198 199 judgment appeal internal contradictions decision issue made findings fact according none evidence administrative file able substantiate liquidation factors used Commission
Answer:  

retrieved examples:

Text: order State aid within meaning provision necessary first aid favouring certain undertakings production certain goods second advantage come State State resources
Answer: ['Class', 'Rule']

Text: contrary exclude priori possibility case present trade union could show party concerned within meaning Article 882 EC relying role collective negotiations effects role national tax measures regarded Commission aid compatible common

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Aut', 'Itpr']
ground truth: Aut
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: considerations far concern information normally prudent diligent private creditor situation close possible local tax office could priori ignore capable justifying General Court’s decision Commission failed take consideration relevant information see effect judgment 24 January 2013 Frucona Košice v Commission C‑7311 P EUC201332 paragraphs 77 78 81
Answer:  

retrieved examples:

Text: General Court indeed held paragraph 63 judgment appeal verification conditions laid Altmark caselaw occurs upstream say examination issue whether measures issue must characterised State aid
Answer: ['Itpr', 'Prec']

Text: Advocate General observed essence point 41 Opinion assessment cannot carried comparing amount airlines subject lower rate ATT requi

In [18]:
test(template, 15)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


query:
Text: regard must observed follows Article 58 Statute Court Justice conjunction Article 1132 Rules Procedure Court Justice appeal appellant may put forward relevant argument provided subjectmatter proceedings General Court changed appeal Case C‑22905 P PKK KNK v Council 2007 ECR I‑439 paragraph 66 Case C‑806 P Herrero Romeu v Commission 2007 ECR I‑10333 paragraph 32
Answer:  

retrieved examples:

Text: Third must confer selective advantage recipient
Answer: Class

Text: contrary exclude priori possibility case present trade union could show party concerned within meaning Article 882 EC relying role collective negotiations effects role national tax measures regarded Commission aid compatible common market would liable undermine social policy objectives induced Court exclude collective agreement issue Albany application Article 851 Treaty
Answer: ['Itpr', 'Rule']

Text: Judicial review compliance European Union legal order ensured seen Article 191 TEU Court Justice courts tribuna

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Rule']
ground truth: ['Prec', 'Rule']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: regard Court repeatedly held action annulment purposes Article 230 EC must available acts adopted institutions whatever nature form intended legal effects capable affecting interests applicant bringing distinct change legal position see inter alia Athinaïki Techniki v Commission paragraph 29 caselaw cited Case C‑36208 P Internationaler Hilfsfonds v Commission 2010 ECR I‑0000 paragraph 51
Answer:  

retrieved examples:

Text: Fourth level compensation needed must determined basis analysis costs typical undertaking well run adequately equipped able meet necessary public service requirements would incurred discharging obligations
Answer: Class

Text: latter case identification economic advantage principle sufficient sup

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

LLM output: ['Itpr', 'Rule']
ground truth: ['Itpr', 'Prec']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Paragraphs 180 213 235 judgment appeal Commission’s objections referred paragraph 64 present judgment directed read legal context referred General Court imply new requirements incompatible caselaw Court Justice
Answer:  

retrieved examples:

Text: absence implementing measures natural legal persons although directly concerned act question would able obtain judicial review act infringed provisions pleading provisions unlawful proceedings initiated national courts
Answer: Itpr

Text: appeal Courts jurisdiction thus confined review findings pleas argued Court First Instance see Case C13692 P Commission v Brazzelli Lualdi Others 1994 ECR I1981 paragraph 59 order 28 June 2001 Case C35299 P Eridania Others v Council 2001 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Rule']
ground truth: Itpr
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Advocate General noted paragraphs 125 131 Opinion General Court merely noted paragraphs 191 195 198 199 judgment appeal internal contradictions decision issue made findings fact according none evidence administrative file able substantiate liquidation factors used Commission
Answer:  

retrieved examples:

Text: Article 1071 TFEU distinguish measures State intervention reference causes aims defines basis effects thus independently techniques used see effect judgments 15 November 2011 Commission Spain v Government Gibraltar United Kingdom C‑10609 P C‑10709 P EUC2011732 paragraphs 87 92 93 28 June 2018 Andres liquidator insolvency Heitkamp BauHolding v Commission C‑20316 P EUC2018505 paragraph 91
Answer: ['Prec', 'Rule']

Text: Ge

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Aut', 'Prec']
ground truth: Aut
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: considerations far concern information normally prudent diligent private creditor situation close possible local tax office could priori ignore capable justifying General Court’s decision Commission failed take consideration relevant information see effect judgment 24 January 2013 Frucona Košice v Commission C‑7311 P EUC201332 paragraphs 77 78 81
Answer:  

retrieved examples:

Text: Article 44 regulation provides Articles 92 93 94 Treaty apply production trade sugar
Answer: Rule

Text: follows Advocate General observed essence point 62 Opinion recovery aid entails restitution advantage procured aid recipient restitution economic benefit recipient may enjoyed result exploiting advantage
Answer: ['Aut', 'Itpr']

Text: must observed

In [19]:
test(template, 30)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


query:
Text: regard must observed follows Article 58 Statute Court Justice conjunction Article 1132 Rules Procedure Court Justice appeal appellant may put forward relevant argument provided subjectmatter proceedings General Court changed appeal Case C‑22905 P PKK KNK v Council 2007 ECR I‑439 paragraph 66 Case C‑806 P Herrero Romeu v Commission 2007 ECR I‑10333 paragraph 32
Answer:  

retrieved examples:

Text: Third must confer selective advantage recipient
Answer: Class

Text: regard recalled Court Justice also held judgment 21 November 2013 Deutsche Lufthansa C‑28412 EUC2013755 paragraph 45 order Court 4 April 2014 Flughafen Lübeck C‑2713 published EUC2014240 paragraph 27 accordance Article 1083 TFEU Commission initiated formal investigation procedure regard measure notified course implementation national court hearing application cessation implementation measure recovery sums already paid required adopt measures necessary order draw appropriate conclusions infringement obligation su

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Rule']
ground truth: ['Prec', 'Rule']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: regard Court repeatedly held action annulment purposes Article 230 EC must available acts adopted institutions whatever nature form intended legal effects capable affecting interests applicant bringing distinct change legal position see inter alia Athinaïki Techniki v Commission paragraph 29 caselaw cited Case C‑36208 P Internationaler Hilfsfonds v Commission 2010 ECR I‑0000 paragraph 51
Answer:  

retrieved examples:

Text: Fourth must distort threaten distort competition see inter alia judgment 16 July 2015 BVVG C‑3914 EUC2015470 paragraph 24
Answer: ['Class', 'Prec']

Text: basis caselaw use regulatory technique cannot enable national tax rules escape outset scrutiny concerning State aid provided FEU Treaty resort

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

LLM output: ['Prec', 'Rule']
ground truth: ['Itpr', 'Prec']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Paragraphs 180 213 235 judgment appeal Commission’s objections referred paragraph 64 present judgment directed read legal context referred General Court imply new requirements incompatible caselaw Court Justice
Answer:  

retrieved examples:

Text: must observed outset Advocate General stated point 47 Opinion requirement selectivity Article 1071 TFEU must clearly distinguished concomitant detection economic advantage Commission identified advantage understood broad sense arising directly indirectly particular measure also required establish advantage specifically benefits one undertakings
Answer: ['Aut', 'Class', 'Rule']

Text: Third must confer selective advantage recipient
Answer: Class

Text: concept ‘State aid’ h

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Aut']
ground truth: Itpr
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Advocate General noted paragraphs 125 131 Opinion General Court merely noted paragraphs 191 195 198 199 judgment appeal internal contradictions decision issue made findings fact according none evidence administrative file able substantiate liquidation factors used Commission
Answer:  

retrieved examples:

Text: Second Advocate General observed point 85 Opinion argument summarised paragraph 71 cannot invalidate finding set paragraph 92 judgment appeal
Answer: Aut

Text: contrary exclude priori possibility case present trade union could show party concerned within meaning Article 882 EC relying role collective negotiations effects role national tax measures regarded Commission aid compatible common market would liable undermine so

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Aut', 'Prec']
ground truth: Aut
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: considerations far concern information normally prudent diligent private creditor situation close possible local tax office could priori ignore capable justifying General Court’s decision Commission failed take consideration relevant information see effect judgment 24 January 2013 Frucona Košice v Commission C‑7311 P EUC201332 paragraphs 77 78 81
Answer:  

retrieved examples:

Text: First must recalled according Court’s settled caselaw classification national measure ‘State aid’ within meaning Article 1071 TFEU requires following conditions fulfilled
Answer: ['Class', 'Prec', 'Rule']

Text: However follows paragraph 37 Article 1062 TFEU require Commission take consideration second fourth Altmark conditions order decide whether St

In [20]:
test(template, 50)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


query:
Text: regard must observed follows Article 58 Statute Court Justice conjunction Article 1132 Rules Procedure Court Justice appeal appellant may put forward relevant argument provided subjectmatter proceedings General Court changed appeal Case C‑22905 P PKK KNK v Council 2007 ECR I‑439 paragraph 66 Case C‑806 P Herrero Romeu v Commission 2007 ECR I‑10333 paragraph 32
Answer:  

retrieved examples:

Text: third must confer selective advantage recipient fourth must distort threaten distort competition see effect judgment 2 September 2010 Commission v Deutsche Post C‑39908 P EUC2010481 paragraph 39 caselaw cited
Answer: Class

Text: Consequently Advocate General observed points 32 34 Opinion even apparent paragraphs 185 188 judgment appeal response fourth plea law action annulment General Court rule existence 2009 serious disturbance Greek economy Hellenic Republic entitled claim Court General Court erred law rejecting argument disturbance justified application Article 1073b TFEU fa

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Rule']
ground truth: ['Prec', 'Rule']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: regard Court repeatedly held action annulment purposes Article 230 EC must available acts adopted institutions whatever nature form intended legal effects capable affecting interests applicant bringing distinct change legal position see inter alia Athinaïki Techniki v Commission paragraph 29 caselaw cited Case C‑36208 P Internationaler Hilfsfonds v Commission 2010 ECR I‑0000 paragraph 51
Answer:  

retrieved examples:

Text: essential characteristic remuneration lies fact consideration service question see effect judgments 11 September 2007 Schwarz GootjesSchwarz C‑7605 EUC2007492 paragraphs 37 38 27 June 2017 Congregación de Escuelas Pías Provincia Betania C‑7416 EUC2017496 paragraphs 45 47
Answer: Prec

Text: imple

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

LLM output: ['Prec', 'Rule']
ground truth: ['Itpr', 'Prec']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Paragraphs 180 213 235 judgment appeal Commission’s objections referred paragraph 64 present judgment directed read legal context referred General Court imply new requirements incompatible caselaw Court Justice
Answer:  

retrieved examples:

Text: must held first Article 4c CS prohibits granting State aid steel coal undertakings without drawing distinction individual aid aid disbursed State aid scheme Article 67 CS refers expressly State aid respect protective measures Commission may authorise pursuant first indent paragraph 2 article favour coal steel undertakings suffer competitive disadvantages general economic policy measures
Answer: Rule

Text: second intervention must liable affect trade Member States
Answer: 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Prec', 'Rule']
ground truth: Itpr
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: Advocate General noted paragraphs 125 131 Opinion General Court merely noted paragraphs 191 195 198 199 judgment appeal internal contradictions decision issue made findings fact according none evidence administrative file able substantiate liquidation factors used Commission
Answer:  

retrieved examples:

Text: must recalled order establish selective nature tax advantage necessary competent national authorities discretionary power grant benefit measure
Answer: Itpr

Text: follows foregoing appropriate criterion establishing selectivity measure issue consists determining whether measure introduces operators light objective pursued general tax system concerned comparable factual legal situation distinction justified nature genera

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


LLM output: ['Aut', 'Prec']
ground truth: Aut
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
query:
Text: considerations far concern information normally prudent diligent private creditor situation close possible local tax office could priori ignore capable justifying General Court’s decision Commission failed take consideration relevant information see effect judgment 24 January 2013 Frucona Košice v Commission C‑7311 P EUC201332 paragraphs 77 78 81
Answer:  

retrieved examples:

Text: However European Union Courts must inter alia establish whether evidence relied factually accurate reliable consistent also whether evidence contains relevant information must taken account order assess complex situation whether capable substantiating conclusions drawn Case C1203 P Commission v Tetra Laval 2005 ECR I987 paragraph 39 Commission v Scott